# Agentic VQA Pipeline — Phi-3.5-Vision-Instruct

Notebook dedicato all'esperimento con **Phi-3.5-Vision-Instruct** (`microsoft/Phi-3.5-vision-instruct`) sul benchmark DUDE.

**Backend**: HuggingFace `transformers` (FP16 nativo, senza Ollama).

Utilizza il profilo `phi35_focused` ottimizzato in base ai risultati di `metrics_summary.csv`:
- **Layout v1** (QUR=0.8503, UR=0.9576) per cause spaziali e layout
- **DocEl CoT v3** (QUR=0.7433, UR=0.9195) per cause di struttura documento e entità/valori
- Famiglie NLP List e NLP Tag evitate (Error Rate >60% e ~6% rispettivamente)

In **Notebook options** abilita:
- Accelerator: **GPU T4 x2**
- Internet: **On**

Aggiungi come Kaggle Dataset il repository e i dati DUDE.

In [ ]:
# ---- PARAMETRI ----
PROJECT_SOURCE = ""  # /kaggle/input/agentic-vqa-pipeline
GIT_REPOSITORY_URL = "https://github.com/matteo-petrelli/Agentic-VQA-Pipeline"  # URL Git pubblico del repository
GIT_REF = "main"

INPUT_JSON_PATH = "/kaggle/input/datasets/matteopetrelli/dude-questions/DUDE_fixed.json"
IMAGE_DIR = "/kaggle/input/datasets/matteopetrelli/dude-train/content/DUDE_train-val-test_binaries/images/train"
OUTPUT_JSON_PATH = "/kaggle/working/unanswerability_diagnostic_results_phi35.json"

# Phi-3.5-Vision specific parameters
HF_MODEL_NAME = "microsoft/Phi-3.5-vision-instruct"
EVIDENCE_GPU = 0
VLM_GPU = 1
ALLOW_SINGLE_GPU_FALLBACK = True
SAMPLING_PERCENTAGE = 0.1
MAX_DOCUMENT_MB = 100
CHUNK_SIZE = 1800
CHUNK_OVERLAP = 200

## 1. Configurazione dell'ambiente Kaggle

Individua il progetto, installa le dipendenze e configura `HF_TOKEN` dai Kaggle Secrets.

## 1.1 Dipendenze Phi-3.5-Vision

Phi-3.5-Vision richiede `transformers==4.43.0` e `flash-attn==2.5.8` (dalla pagina HuggingFace ufficiale).
Queste versioni vengono pinnate **prima** del setup principale per evitare conflitti.

In [ ]:
# Pin Phi-3.5-Vision compatible versions (from HuggingFace model page)
# Must run BEFORE setup_environment to avoid loading wrong transformers version
import subprocess, sys

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.43.0",
    "flash_attn==2.5.8",
    "accelerate==0.30.0",
], check=True)
print("Phi-3.5-Vision dependencies installed.")

In [ ]:
from pathlib import Path

WORKING_DIR = Path("/kaggle/working") if Path("/kaggle").exists() else Path.cwd()

# Find or clone the project
import os, sys
# Minimal bootstrap: find project before importing kaggle_utils
_clone_dir = WORKING_DIR / "Agentic-VQA-Pipeline"
for _candidate in ([Path(PROJECT_SOURCE).expanduser()] if PROJECT_SOURCE else []) + [_clone_dir, Path.cwd()]:
    if _candidate and (_candidate / "kaggle_utils.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    if GIT_REPOSITORY_URL:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, GIT_REPOSITORY_URL, str(_clone_dir)], check=True)
        sys.path.insert(0, str(_clone_dir))

from kaggle_utils import find_project, setup_environment

PROJECT_DIR = find_project(PROJECT_SOURCE, GIT_REPOSITORY_URL, GIT_REF, WORKING_DIR)
setup_environment(PROJECT_DIR)

## 2. Rilevamento e assegnazione delle GPU

PyTorch vede entrambe le GPU. DOTS/GLiNER useranno `cuda:0`; Phi-3.5-Vision userà `cuda:1`.

In [ ]:
from kaggle_utils import detect_gpus

EVIDENCE_DEVICE, VLM_GPU_DEVICE = detect_gpus(EVIDENCE_GPU, VLM_GPU, ALLOW_SINGLE_GPU_FALLBACK)
HF_VLM_DEVICE = f"cuda:{VLM_GPU_DEVICE}" if isinstance(VLM_GPU_DEVICE, int) else VLM_GPU_DEVICE
print(f"Evidence device: {EVIDENCE_DEVICE}")
print(f"VLM device: {HF_VLM_DEVICE}")

## 3. Caricamento motore DOTS/GLiNER + configurazione backend Transformers

A differenza dei notebook Ollama, qui configuriamo il backend **`transformers`** per caricare
Phi-3.5-Vision direttamente da HuggingFace in FP16 sulla GPU dedicata.

In [ ]:
import json
import config

# Dataset and output
config.INPUT_JSON_PATH = INPUT_JSON_PATH
config.IMAGE_DIR = IMAGE_DIR
config.OUTPUT_JSON_PATH = OUTPUT_JSON_PATH
config.EVIDENCE_DEVICE = EVIDENCE_DEVICE
config.SAMPLING_PERCENTAGE = SAMPLING_PERCENTAGE
config.VLM_NUM_CTX = 8192
config.VLM_MAX_TOKENS = 1536

# Switch to transformers backend (instead of Ollama)
config.VLM_BACKEND = "transformers"
config.HF_VLM_MODEL_NAME = HF_MODEL_NAME
config.HF_VLM_DEVICE = HF_VLM_DEVICE
config.HF_VLM_DTYPE = "float16"
config.HF_VLM_CACHE_DIR = "/tmp/hf_cache"

# Prompt profile (auto-selected via MODEL_PROFILE_MAP for phi3.5)
config.PROMPT_PROFILE = "phi35_focused"

# Set OLLAMA_VLM to a descriptive name for logging/profile resolution
config.OLLAMA_VLM = "phi3.5-vision"

from diagnostic_agent.engine import DocumentEngine

engine = DocumentEngine()
print("DocumentEngine caricato correttamente (DOTS + GLiNER).")

## 4. Caricamento del modello Phi-3.5-Vision (HuggingFace Transformers)

Carica esplicitamente il modello VLM sulla GPU dedicata prima di iniziare l'esperimento.
Il caricamento lazy avverrebbe comunque al primo `infer()`, ma il caricamento eager
consente di verificare che il modello funzioni prima di lanciare il run completo.

In [ ]:
# Eager loading of Phi-3.5-Vision
engine.setup_transformers_vlm()
print(f"\nPhi-3.5-Vision loaded successfully on {config.HF_VLM_DEVICE}.")
print(f"Backend: {config.VLM_BACKEND}")
print(f"Model: {config.HF_VLM_MODEL_NAME}")

## 4.1 Verifica del profilo attivo

Conferma che il profilo `phi35_focused` è stato selezionato e mostra la mappa cause → prompt.

In [ ]:
from diagnostic_agent.profiles import resolve_prompt_profile

active_profile = resolve_prompt_profile(config.OLLAMA_VLM, config.PROMPT_PROFILE)
print(f"Active profile: {active_profile.name}")
print(f"Answerer prompt: {active_profile.answerer_prompt}")
print(f"Verifier prompt: {active_profile.verifier_prompt}")
print("\nCause → Prompt mapping:")
for cause, prompt in active_profile.cause_prompts.items():
    print(f"  {cause.value:35s} → {prompt}")

## 5. Esperimento completo (con checkpointing e sampling)

Il run completo usa `run_experiments.main()` che include:
- **Checkpointing**: salva il risultato dopo ogni domanda, riprende da dove si era fermato
- **Sampling**: rispetta `SAMPLING_PERCENTAGE` per limitare le domande elaborate
- **Profile**: usa automaticamente `phi35_focused` grazie al MODEL_PROFILE_MAP
- **Backend**: usa `transformers` (niente Ollama)

In [ ]:
from run_experiments import main as run_full_experiment

run_full_experiment(model_name=config.OLLAMA_VLM, engine=engine)

## 6. Salvataggio dei risultati

Esporta risultati in JSON, CSV e TXT.

In [ ]:
from kaggle_utils import export_results

export_results(
    OUTPUT_JSON_PATH,
    smoke_result=None,
    smoke_question=None,
    smoke_image_paths=None,
    run_full=True,
    working_dir=WORKING_DIR,
)

### Pulizia memoria

Libera la memoria GPU al termine dell'esperimento.

In [ ]:
import gc
import torch

# Cleanup VLM model
if hasattr(engine, '_hf_vlm_model') and engine._hf_vlm_model is not None:
    del engine._hf_vlm_model
    engine._hf_vlm_model = None
if hasattr(engine, '_hf_vlm_processor') and engine._hf_vlm_processor is not None:
    del engine._hf_vlm_processor
    engine._hf_vlm_processor = None
torch.cuda.empty_cache()
gc.collect()
print("Memoria GPU liberata.")